<a href="https://colab.research.google.com/github/jillianb2m/tcc-futebol-analise/blob/develop/tcc_ml_futebol.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Installs & Imports

## 1.1. Installs

In [ ]:
!pip install pingouin > /dev/null

## 1.2. Imports

In [ ]:
import pandas as pd
import numpy as np
import traceback as tb
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from matplotlib.patches import FancyBboxPatch, Patch
from scipy.stats import zscore
import scipy.cluster.hierarchy as sch
import pingouin as pg
import plotly.express as px
import os

# 2. Configuração

In [ ]:
# 5.2. Função main
%cd /content/tcc-futebol-analise
BASE_DIR = os.getcwd()

CONFIG = {
    'MIN_JOGOS': 5,
    'DATASET_PATH': os.path.join(BASE_DIR, 'datasets', 'processed', 'brasileirao_opta_final.csv'),
    'N_CLUSTERS_MAX': 10,
    'METODO_LINKAGE': 'complete',
    'METODO_PADRONIZAR': 'sklearn',
    'RANDOM_STATE': 100
}

# 5.1.1. Inicialização e gerar dataset base
regras_col_agregacao = {
    'rating': 'mean',
    'minutesPlayed': 'sum',
    'totalShots': 'mean',
    'onTargetScoringAttempt': 'mean',
    'bigChanceCreated': 'mean',
    'expectedGoals': 'mean',
    'goals': 'sum',
    'goalAssist': 'sum',
    'totalPass': 'mean',
    'accuratePass': 'mean',
    'keyPass': 'mean',
    'touches': 'mean',
    'possessionLostCtrl': 'mean',
    'totalTackle': 'mean',
    'interceptionWon': 'mean',
    'ballRecovery': 'mean',
    'aerialWon': 'mean',
    'Avg_X': 'mean',
    'Avg_Y': 'mean'
}

# 5.1.1. Inicialização e gerar dataset base
variaveis_esperadas = [
    'totalPass',
    'onTargetScoringAttempt',
    'bigChanceCreated',
    'expectedGoals',
    'accuratePass',
    'touches',
    'possessionLostCtrl',
    'ballRecovery',
    'totalShots',
    'goals',
    'totalTackle',
    'interceptionWon',
    'keyPass',
    'goalAssist',
    'aerialWon',
    'Avg_X',
    'Avg_Y'
]

# 5.1.2. Análise Exploratória
mapa_variaveis = {
    'rating': 'Avaliação',
    'minutesPlayed': 'Minutos Jogados',
    'totalPass': 'Total de passes',
    'onTargetScoringAttempt': 'Finalizações no alvo',
    'bigChanceCreated': 'Grandes chances criadas',
    'expectedGoals': 'Gols esperados (xG)',
    'accuratePass': 'Passes certos',
    'touches': 'Toques na bola',
    'possessionLostCtrl': 'Perdas de posse',
    'ballRecovery': 'Recuperações de bola',
    'totalShots': 'Finalizações',
    'goals': 'Gols',
    'totalTackle': 'Desarmes',
    'interceptionWon': 'Interceptações',
    'keyPass': 'Passes-chave',
    'goalAssist': 'Assistências',
    'aerialWon': 'Duelos aéreos ganhos',
    'Avg_X': 'Posição Horizontal',
    'Avg_Y': 'Posição Vertical'
}

# 5.1.2. Análise Exploratória
texto_legenda = (
    "Horizontal (Posição X):\n"
    "• 0 = Lado Esquerdo\n"
    "• 50 = Centro do Campo\n"
    "• 100 = Lado Direito\n\n"
    "Vertical (Posição Y):\n"
    "• 0 = Linha de Ataque\n"
    "• 50 = Linha do Meio Campo\n"
    "• 100 = Linha de Defesa\n\n"
    "Exemplos:\n"
    "• Goleiro: (50, 90)\n"
    "• Zagueiro: (50, 75)\n"
    "• Volante: (50, 60)\n"
    "• Meia: (50, 40)\n"
    "• Atacante: (50, 20)"
)

metricas_por_posicao = {
    'Goleiro': [
        'ballRecovery',
        'aerialWon',
        'totalPass',
        'accuratePass'
    ],
    'Zagueiro': [
        'aerialWon',
        'interceptionWon',
        'totalTackle',
        'ballRecovery'
    ],
    'Volante': [
        'totalTackle',
        'interceptionWon',
        'ballRecovery',
        'totalPass',
        'accuratePass'
    ],
    'Meia Central': [
        'keyPass',
        'goalAssist',
        #'bigChanceCreated', -- RETORNAR SE NÃO FIZER DIFERENÇA NOS CLUSTERS
        'goals',
        'totalShots',
        # 'onTargetScoringAttempt', -- RETORNAR SE NÃO FIZER DIFERENÇA NOS CLUSTERS
        # 'expectedGoals',  -- RETORNAR SE NÃO FIZER DIFERENÇA NOS CLUSTERS
        'totalPass',
        'accuratePass',
        'touches'
    ]
}

# 6.1.2. Definição de benchmarks

regras_gap_agregacao = {
  'rating': 'mean',
  'goals': 'sum',
  'totalTackle': 'mean',
  'interceptionWon': 'mean',
  'ballRecovery': 'mean',
  'keyPass': 'mean',
  'goalAssist': 'sum',
  'totalPass': 'mean',
  'accuratePass': 'mean',
  'aerialWon': 'mean',
  'onTargetScoringAttempt': 'mean',
  'bigChanceCreated': 'mean',
  'expectedGoals': 'mean'
}

# 6.1.7. Perfil recomendado
mapeamento_perfis = {
    'Goleiro': {
        'ballRecovery': 'Defesa',
        'aerialWon': 'Defesa',
        'totalPass': 'Saída de Jogo',
        'accuratePass': 'Saída de Jogo'
    },
    'Zagueiro': {
        'aerialWon': 'Aéreo',
        'interceptionWon': 'Bola no Chão',
        'totalTackle': 'Bola no Chão',
        'ballRecovery': 'Bola no Chão'
    },
    'Volante': {
        'totalTackle': 'Defensivo',
        'interceptionWon': 'Defensivo',
        'ballRecovery': 'Defensivo',
        'totalPass': 'Organizador',
        'accuratePass': 'Organizador'
    },
    'Meia Central': {
        'keyPass': 'Criador',
        'goalAssist': 'Criador',
        'goals': 'Finalizador',
        'totalShots': 'Finalizador',
        'expectedGoals': 'Finalizador'
    },
    'Meia Direita': {
        'keyPass': 'Criador',
        'goalAssist': 'Criador',
        'totalPass': 'Organizador'
    },
    'Meia Esquerda': {
        'keyPass': 'Criador',
        'goalAssist': 'Criador',
        'totalPass': 'Organizador'
    },
    'Meia Atacante': {
        'keyPass': 'Criador',
        'goalAssist': 'Criador',
        'goals': 'Finalizador',
        'expectedGoals': 'Finalizador'
    },
    'Ponta Direita': {
        'goals': 'Finalizador',
        'keyPass': 'Criador',
        'goalAssist': 'Criador',
        'expectedGoals': 'Finalizador'
    },
    'Ponta Esquerda': {
        'goals': 'Finalizador',
        'keyPass': 'Criador',
        'goalAssist': 'Criador',
        'expectedGoals': 'Finalizador'
    },
    'Lateral Direito': {
        'totalPass': 'Organizador',
        'totalTackle': 'Defensivo',
        'ballRecovery': 'Defensivo'
    },
    'Lateral Esquerdo': {
        'totalPass': 'Organizador',
        'totalTackle': 'Defensivo',
        'ballRecovery': 'Defensivo'
    }
}

# 6.1.5. Cálcular pontuação
pesos_por_posicao = {
    'Goleiro': {
        'ballRecovery': 3,
        'aerialWon': 2,
        'totalPass': 2,
        'accuratePass': 2,
        'rating': 2
    },
    'Zagueiro': {
        'aerialWon': 3,
        'interceptionWon': 3,
        'totalTackle': 2,
        'ballRecovery': 2,
        'rating': 2
    },
    'Volante': {
        'totalTackle': 3,
        'interceptionWon': 2,
        'ballRecovery': 2,
        'totalPass': 2,
        'accuratePass': 1,
        'rating': 2
    },
    'Meia Central': {
        'keyPass': 3,
        'goalAssist': 3,
        'goals': 2,
        'totalShots': 1,
        'totalPass': 1,
        'accuratePass': 1,
        'rating': 2
    },
    'Meia Direita': {
        'keyPass': 2,
        'goalAssist': 2,
        'totalPass': 2,
        'accuratePass': 1,
        'rating': 2
    },
    'Meia Esquerda': {
        'keyPass': 2,
        'goalAssist': 2,
        'totalPass': 2,
        'accuratePass': 1,
        'rating': 2
    },
    'Meia Atacante': {
        'keyPass': 3,
        'goalAssist': 3,
        'goals': 3,
        'expectedGoals': 2,
        'rating': 2
    },
    'Ponta Direita': {
        'goals': 3,
        'keyPass': 2,
        'goalAssist': 2,
        'expectedGoals': 2,
        'rating': 2
    },
    'Ponta Esquerda': {
        'goals': 3,
        'keyPass': 2,
        'goalAssist': 2,
        'expectedGoals': 2,
        'rating': 2
    },
    'Lateral Direito': {
        'totalPass': 2,
        'accuratePass': 2,
        'totalTackle': 2,
        'ballRecovery': 2,
        'rating': 2
    },
    'Lateral Esquerdo': {
        'totalPass': 2,
        'accuratePass': 2,
        'totalTackle': 2,
        'ballRecovery': 2,
        'rating': 2
    }
}

/content/tcc-futebol-analise


# 3. Github

## 3.1. Clone

In [ ]:
%run "/content/drive/MyDrive/Colab Notebooks/tcc-clone-repo.py"

## 3.2. Commit

In [ ]:
#%run "/content/drive/MyDrive/Colab Notebooks/tcc-commit-github.py"

# 4. DataSet

## 4.1. Coleta dados Sofascore

In [ ]:
# %run "/content/tcc-futebol-analise/src/scripts/coleta_dados_sofascore.py"

## 4.2. Tratamento dos dados

In [ ]:
# %run "/content/tcc-futebol-analise/src/scripts/preparar_base.py"

##4.2. Análise do Datase

In [ ]:

OUTPUT_PATH = os.path.join(BASE_DIR, 'distribuicao_posicoes.png')

def carregar_e_filtrar_dados(dataset_path, min_jogos=5):

    print("=== CARREGANDO E FILTRANDO DADOS ===")

    dataframe = pd.read_csv(dataset_path)
    print(f"Dataset carregado: {len(dataframe)} registros")

    df_filtro = dataframe.copy()

    colunas_existentes = [col for col in regras_col_agregacao.keys() if col in df_filtro.columns]
    agregacoes_filtradas = {k: v for k, v in regras_col_agregacao.items() if k in colunas_existentes}

    df_agregado = df_filtro.groupby(['Jogador', 'Time', 'Posicao_Real']).agg(agregacoes_filtradas).reset_index()
    print(f"Após agregação: {len(df_agregado)} jogadores únicos")

    minutos_minimos = min_jogos * 90
    df_filtrado = df_agregado[df_agregado['minutesPlayed'] >= minutos_minimos].copy()
    print(f"Após filtro de {min_jogos} jogos ({minutos_minimos} min): {len(df_filtrado)} jogadores")

    return df_filtrado

def gerar_grafico_distribuicao_posicoes(dados_filtrados):

    print("\n=== DISTRIBUIÇÃO DE JOGADORES POR POSIÇÃO ===")

    if 'Posicao_Real' not in dados_filtrados.columns:
        print("Erro: Coluna 'Posicao_Real' não encontrada no DataFrame.")
        return None

    distribuicao = dados_filtrados['Posicao_Real'].value_counts().sort_values(ascending=False)

    total_jogadores = len(dados_filtrados)
    percentuais = (distribuicao / total_jogadores * 100).round(1)

    print(f"Total de jogadores: {total_jogadores}")
    print("\nDistribuição por posição:")
    for posicao, count in distribuicao.items():
        print(f"{posicao}: {count} jogadores ({percentuais[posicao]}%)")

    fig, ax = plt.subplots(figsize=(14, 7))

    n_barras = len(distribuicao)
    cores = plt.cm.viridis(np.linspace(0, 1, n_barras))

    bars = ax.bar(range(n_barras), distribuicao.values,
                   color=cores, alpha=0.8, edgecolor='darkblue', linewidth=1.5)

    for i, (bar, count, pct) in enumerate(zip(bars, distribuicao.values, percentuais)):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{count}\n({pct}%)',
                ha='center', va='bottom', fontsize=10, fontweight='bold')

    ax.set_xlabel('Posições', fontsize=14, fontweight='bold')
    ax.set_ylabel('Número de Jogadores', fontsize=14, fontweight='bold')
    ax.set_title('Distribuição de Jogadores por Posição',
                fontsize=16, fontweight='bold')
    ax.set_xticks(range(len(distribuicao)))
    ax.set_xticklabels(distribuicao.index, rotation=45, ha='right', fontsize=11)

    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.savefig(OUTPUT_PATH, dpi=300, bbox_inches='tight')
    print(f"\nGráfico salvo em: {OUTPUT_PATH}")
    #plt.show()
    plt.close()

    return distribuicao

if __name__ == "__main__":
    dados_filtrados = carregar_e_filtrar_dados(CONFIG['DATASET_PATH'], min_jogos=5)

    if dados_filtrados is not None:
        gerar_grafico_distribuicao_posicoes(dados_filtrados)

    print("\nScript concluído com sucesso!")

=== CARREGANDO E FILTRANDO DADOS ===
Dataset carregado: 35319 registros
Após agregação: 5632 jogadores únicos
Após filtro de 5 jogos (450 min): 1286 jogadores

=== DISTRIBUIÇÃO DE JOGADORES POR POSIÇÃO ===
Total de jogadores: 1286

Distribuição por posição:
Meia Central: 260 jogadores (20.2%)
Volante: 238 jogadores (18.5%)
Meia Direita: 179 jogadores (13.9%)
Meia Esquerda: 172 jogadores (13.4%)
Zagueiro: 172 jogadores (13.4%)
Meia Atacante: 86 jogadores (6.7%)
Goleiro: 65 jogadores (5.1%)
Lateral Esquerdo: 39 jogadores (3.0%)
Lateral Direito: 39 jogadores (3.0%)
Ponta Direita: 22 jogadores (1.7%)
Ponta Esquerda: 14 jogadores (1.1%)

Gráfico salvo em: /content/tcc-futebol-analise/distribuicao_posicoes.png

Script concluído com sucesso!


# 5. Clustering

## 5.1. Classe Clustering

### 5.1.1. Inicialização e gerar dataset base

In [ ]:
class ClusterBrasileirao:
  def __init__(self, dataset_path):
    self.dataframe = pd.read_csv(dataset_path)
    self.dados_cluster = None
    self.variaveis_metricas = None
    self.dados_padronizados = None
    self.scaler = None
    self.centroides = None
    self.df_anova = None

  def gerar_base(self, min_jogos=5, posicao_alvo=None):
    df_filtro_alvo = self.dataframe.copy()

    if posicao_alvo:
        df_filtro_alvo = df_filtro_alvo[df_filtro_alvo['Posicao_Real'] == posicao_alvo].copy()

    colunas_existentes = [col for col in regras_col_agregacao.keys() if col in df_filtro_alvo.columns]
    agregacoes_filtradas = {k: v for k, v in regras_col_agregacao.items() if k in colunas_existentes}

    df_agregado_alvo = df_filtro_alvo.groupby(['Jogador', 'Time', 'Posicao_Real']).agg(agregacoes_filtradas).reset_index()

    df_alvo = df_agregado_alvo[df_agregado_alvo['minutesPlayed'] >= min_jogos * 90].copy()

    # Tratar outliers com Winsorização (capping) para métricas específicas
    # REMOVER SE NÃO FIZER DIFERENÇA NA CRIAÇÃO DOS CLUSTERS
    if 'goalAssist' in df_alvo.columns:
        percentile_95 = df_alvo['goalAssist'].quantile(0.95)
        df_alvo['goalAssist'] = df_alvo['goalAssist'].clip(upper=percentile_95)

    if 'goals' in df_alvo.columns:
        percentile_95 = df_alvo['goals'].quantile(0.95)
        df_alvo['goals'] = df_alvo['goals'].clip(upper=percentile_95)

    if posicao_alvo and posicao_alvo in metricas_por_posicao:
        v_metricas = [v for v in metricas_por_posicao[posicao_alvo] if v in df_alvo.columns]
        print(f"Usando métricas específicas para {posicao_alvo}: {v_metricas}")
    else:
        v_metricas = [v for v in variaveis_esperadas if v in df_alvo.columns]

    colunas_manter = ['Jogador', 'Time', 'Posicao_Real', 'minutesPlayed', 'rating'] + v_metricas
    colunas_manter = [c for c in colunas_manter if c in df_alvo.columns]

    self.dados_cluster = df_alvo[colunas_manter].copy()
    self.variaveis_metricas = v_metricas
    dados_metricos = self.dados_cluster[v_metricas].copy()

    if posicao_alvo:
        print(f"Total de jogadores clusterizados ({posicao_alvo}): {len(self.dados_cluster)}")
    else:
        print(f"Total de jogadores clusterizados: {len(self.dados_cluster)}")
        print(f"Distribuição por posição:")
        print(self.dados_cluster['Posicao_Real'].value_counts())

    return dados_metricos

### 5.1.2. Análise Exploratória

In [ ]:
def explorar_variaveis_dataframe(self):
  print("********* Análise Exploratória *********")

  pd.set_option('display.max_columns', None)
  pd.set_option('display.expand_frame_repr', False)

  print(self.dados_cluster[self.variaveis_metricas].describe().loc[['mean', 'std', 'min', 'max']].T.round(2))
  print("\n")

  posicao_atual = self.dados_cluster['Posicao_Real'].iloc[0] if 'Posicao_Real' in self.dados_cluster.columns else None
  todas_posicoes_iguais = self.dados_cluster['Posicao_Real'].nunique() == 1 if 'Posicao_Real' in self.dados_cluster.columns else False

  if todas_posicoes_iguais and posicao_atual in metricas_por_posicao:
    variaveis_plot = [v for v in metricas_por_posicao[posicao_atual] if v in self.variaveis_metricas]
    titulo = f'Distribuição das Variáveis - {posicao_atual}'
  else:
    variaveis_plot = self.variaveis_metricas.copy()
    titulo = 'Distribuição das Variáveis de Desempenho'

  n_vars = len(variaveis_plot)
  if n_vars <= 3:
    n_cols = n_vars
    n_rows = 1
  elif n_vars <= 6:
    n_cols = 3
    n_rows = 2
  else:
    n_cols = 3
    n_rows = 3

  fig, axes = plt.subplots(n_rows, n_cols, figsize=(24, 8 * n_rows))
  fig.suptitle(titulo, fontsize=18)

  if n_rows == 1:
    axes = axes.reshape(1, -1)

  for idx, var in enumerate(variaveis_plot):
    row = idx // n_cols
    col = idx % n_cols

    dados_melt = pd.melt(self.dados_cluster[[var]])
    dados_melt['variable'] = dados_melt['variable'].replace(mapa_variaveis)
    sns.boxplot(x='variable', y='value', hue='variable', data=dados_melt, ax=axes[row, col], palette='viridis', legend=False)
    axes[row, col].set_title(mapa_variaveis.get(var, var), fontsize=14)
    axes[row, col].set_ylabel('Valores', fontsize=12)
    axes[row, col].set_xlabel('', fontsize=12)
    axes[row, col].tick_params(axis='x', rotation=45)

  for idx in range(len(variaveis_plot), n_rows * n_cols):
    row = idx // n_cols
    col = idx % n_cols
    axes[row, col].axis('off')

  if not todas_posicoes_iguais and n_rows * n_cols > len(variaveis_plot):
    last_row = n_rows - 1
    last_col = n_cols - 1
    axes[last_row, last_col].set_title('Legenda - Posicionamento em Campo', fontsize=14)

    rect = FancyBboxPatch((0.1, 0.15), 0.8, 0.7,
                          transform=axes[last_row, last_col].transAxes,
                          facecolor='lightblue', alpha=0.9,
                          edgecolor='navy', linewidth=2.0,
                          boxstyle='round,pad=0.5')
    axes[last_row, last_col].add_patch(rect)

    axes[last_row, last_col].text(0.15, 0.50, texto_legenda, transform=axes[last_row, last_col].transAxes,
                  fontsize=14, ha='left', va='center')

  plt.tight_layout()
  plt.savefig(os.path.join(BASE_DIR, 'boxplot_analise_exploratoria.png'), dpi=300, bbox_inches='tight')
  #plt.show()
  plt.close()

  print("\n")

ClusterBrasileirao.explorar_variaveis_dataframe = explorar_variaveis_dataframe

### 5.1.3. Padronização dos Dados (ZScore ou StandardScaler)

In [ ]:
def padronizar_dados(self, metodo='zscore'):
  dados_metricos = self.dados_cluster[self.variaveis_metricas].copy()

  if metodo == 'zscore':
    self.dados_padronizados = dados_metricos.apply(zscore, ddof=1)
    print("Padronização Z-Score aplicada")
  else:
    self.scaler = StandardScaler()
    self.dados_padronizados = pd.DataFrame(self.scaler.fit_transform(dados_metricos),
                                            columns=dados_metricos.columns,
                                            index=dados_metricos.index
                                          )
    print("Padronização StandardScaler aplicada")
  return self.dados_padronizados

ClusterBrasileirao.padronizar_dados = padronizar_dados

### 5.1.4. Análise quantidade de clusters

In [ ]:
def analisar_clusters(self, max_clusters=8, random_state=100):
  print("Analisando número ideal de clusters\n")

  k_elbow = self.analisar_elbow(max_clusters, random_state)
  k_silhouette = self.analisar_silhouette(max_clusters, random_state)

  print(f"Elbow: {k_elbow}")
  print(f"Silhouette: {k_silhouette}")

  if k_elbow > k_silhouette:
    return k_elbow
  else:
    return k_silhouette

ClusterBrasileirao.analisar_clusters = analisar_clusters

#### 5.1.4.1 Método Elbow

In [ ]:
def analisar_elbow(self, max_clusters=8, random_state=100):
  elbow = []
  K = list(range(1, max_clusters + 1))

  for k in K:
    kmeans = KMeans(
        n_clusters=k,
        init='k-means++',
        n_init=10,
        random_state=random_state
    ).fit(self.dados_padronizados)

    elbow.append(kmeans.inertia_)

  plt.figure(figsize=(12, 8), dpi=600)
  plt.plot(K, elbow, marker='o', linewidth=2, markersize=8)
  plt.xlabel('Nº Clusters', fontsize=16)
  plt.ylabel('WCSS', fontsize=16)
  plt.title('Método de Elbow', fontsize=16)
  plt.grid(True, alpha=0.3)
  plt.tight_layout()
  plt.savefig(os.path.join(BASE_DIR, 'metodo_elbow.png'), dpi=300, bbox_inches='tight')
  #plt.show()
  plt.close()

  points = np.array(list(zip(K, elbow)))

  linha_inicio = points[0]
  linha_fim = points[-1]

  x1, y1 = linha_inicio
  x2, y2 = linha_fim

  distancias = []
  for ponto in points:
    x0, y0 = ponto

    distancia = abs(
        (y2 - y1)*x0 - (x2 - x1)*y0 + x2*y1 - y2*x1
    ) / np.sqrt((y2 - y1)**2 + (x2 - x1)**2)

    distancias.append(distancia)

  cotovelo_sugerido = np.argmax(distancias) + 1

  print(f"Nº sugerido de clusters (cotovelo): {cotovelo_sugerido}")
  return cotovelo_sugerido

ClusterBrasileirao.analisar_elbow = analisar_elbow

#### 5.1.4.2. Método Silhueta

In [ ]:
def analisar_silhouette(self, max_clusters=8, random_state=100):
  scores = []
  K = list(range(2, max_clusters + 1))

  for k in K:
    kmeans = KMeans(
        n_clusters=k,
        init='k-means++',
        n_init=10,
        random_state=random_state
    )

    labels = kmeans.fit_predict(self.dados_padronizados)
    score = silhouette_score(self.dados_padronizados, labels)
    scores.append(score)

  plt.figure(figsize=(12, 8), dpi=600)
  plt.plot(K, scores, marker='o', linewidth=2, markersize=8)
  plt.xlabel('Nº de Clusters', fontsize=16)
  plt.ylabel('Silhouette Score', fontsize=16)
  plt.title('Análise do Índice de Silhouette', fontsize=16)
  plt.grid(True, alpha=0.3)
  plt.tight_layout()
  plt.savefig(os.path.join(BASE_DIR, 'metodo_silhouette.png'), dpi=300, bbox_inches='tight')
  #plt.show()
  plt.close()

  melhor_k = K[scores.index(max(scores))]

  print(f"Nº sugerido de clusters (Silhouette): {melhor_k}")
  return melhor_k

ClusterBrasileirao.analisar_silhouette = analisar_silhouette

### 5.1.5. Clustering Hierárquico

In [ ]:
def aplicar_hierarquico(self, n_clusters=3, metodo_linkage='average', metrica_distancia='euclidean'):
  print(f"\n********** Algoritmo Hierárquico ({metodo_linkage} linkage) **********")

  plt.figure(figsize=(16, 8), dpi=600)
  linkage_matrix = sch.linkage(self.dados_padronizados, method=metodo_linkage, metric=metrica_distancia)

  labels_jogadores = [f"{jog}"[:15] for jog in self.dados_cluster['Jogador'].values]
  altura_corte_color = linkage_matrix[-(n_clusters-1), 2] * 0.95
  dendrogram = sch.dendrogram(linkage_matrix, labels=labels_jogadores, color_threshold=altura_corte_color)

  plt.title(f'Dendrograma - {metodo_linkage.title()} Linkage', fontsize=16)
  plt.xlabel('Jogadores', fontsize=16)
  plt.ylabel(f'Distância {metrica_distancia.title()}', fontsize=16)
  plt.axhline(y=12, color='red', linestyle='--')
  plt.axhline(y=altura_corte_color, color='red', linestyle='--', linewidth=2)
  plt.xticks(rotation=90)
  plt.tight_layout()
  plt.savefig(os.path.join(BASE_DIR, f'dendrograma_{metodo_linkage}.png'), dpi=300, bbox_inches='tight')
  #plt.show()
  plt.close()

  cluster_hierarquico = AgglomerativeClustering(n_clusters=n_clusters,
                                                metric=metrica_distancia,
                                                linkage=metodo_linkage
                                                )

  indicadores_cluster = cluster_hierarquico.fit_predict(self.dados_padronizados)
  self.dados_cluster[f'cluster_hierarchical_{metodo_linkage}'] = indicadores_cluster
  self.dados_cluster[f'cluster_hierarchical_{metodo_linkage}'] = self.dados_cluster[f'cluster_hierarchical_{metodo_linkage}'].astype('category')

  coeficientes = [y[1] for y in dendrogram['dcoord']]
  print(f"Coeficientes de aglomeração ({metodo_linkage}): {coeficientes[:5]}...")

  print(f"\nDistribuição dos clusters ({metodo_linkage}):")
  for i in range(n_clusters):
    count = (indicadores_cluster == i).sum()
    percent = (count / len(indicadores_cluster)) * 100
    print(f"Cluster {i}: {count} jogadores ({percent:.1f}%)")

  return indicadores_cluster

ClusterBrasileirao.aplicar_hierarquico = aplicar_hierarquico

### 5.1.6. Clustering K-Means

In [ ]:
def aplicar_kmeans(self, n_clusters=3, init_method='random', random_state=100):
  print(f"\n********** Algoritmo K-MEANS (k={n_clusters}) **********")

  kmeans = KMeans(
    n_clusters=n_clusters,
    init='k-means++',
    n_init=50,
    random_state=random_state
  ).fit(self.dados_padronizados)

  kmeans_clusters = kmeans.labels_
  self.dados_cluster['cluster_kmeans'] = kmeans_clusters
  self.dados_cluster['cluster_kmeans'] = self.dados_cluster['cluster_kmeans'].astype('category')

  centroides_originais = self.scaler.inverse_transform(kmeans.cluster_centers_)
  self.centroides = pd.DataFrame(centroides_originais, columns=self.variaveis_metricas)

  self.centroides.index.name = 'cluster'

  print("\nCentroides dos clusters:")
  print(self.centroides.round(3))

  nomes_clusters = self.identificar_perfis_clusters_centroides()

  if len(self.variaveis_metricas) >= 2 and len(self.dados_cluster) > 50:
    try:
      plt.figure(figsize=(10, 8), dpi=300)

      posicao = self.dados_cluster['Posicao_Real'].iloc[0]

      if posicao == 'Goleiro':
        if 'ballRecovery' in self.variaveis_metricas and 'aerialWon' in self.variaveis_metricas:
          var1, var2 = 'ballRecovery', 'aerialWon'
        elif 'totalPass' in self.variaveis_metricas and 'ballRecovery' in self.variaveis_metricas:
          var1, var2 = 'totalPass', 'ballRecovery'
        else:
          var1, var2 = self.variaveis_metricas[0], self.variaveis_metricas[1]

      elif posicao == 'Zagueiro':
        if 'aerialWon' in self.variaveis_metricas and 'interceptionWon' in self.variaveis_metricas:
          var1, var2 = 'aerialWon', 'interceptionWon'
        elif 'totalTackle' in self.variaveis_metricas and 'aerialWon' in self.variaveis_metricas:
          var1, var2 = 'totalTackle', 'aerialWon'
        else:
          var1, var2 = self.variaveis_metricas[0], self.variaveis_metricas[1]

      elif posicao == 'Volante':
        if 'totalTackle' in self.variaveis_metricas and 'totalPass' in self.variaveis_metricas:
          var1, var2 = 'totalTackle', 'totalPass'
        elif 'interceptionWon' in self.variaveis_metricas and 'totalPass' in self.variaveis_metricas:
          var1, var2 = 'interceptionWon', 'totalPass'
        else:
          var1, var2 = self.variaveis_metricas[0], self.variaveis_metricas[1]

      else:
        if 'keyPass' in self.variaveis_metricas and 'goals' in self.variaveis_metricas:
          var1, var2 = 'keyPass', 'goals'
        elif 'totalPass' in self.variaveis_metricas and 'keyPass' in self.variaveis_metricas:
          var1, var2 = 'totalPass', 'keyPass'
        else:
          var1, var2 = self.variaveis_metricas[0], self.variaveis_metricas[1]

      sns.scatterplot(
        data=self.dados_cluster,
        x=var1,
        y=var2,
        hue='cluster_kmeans',
        palette='viridis',
        s=100,
        legend='full'
      )

      centroides_plot = self.centroides[[var1, var2]].copy()

      plt.scatter(
        centroides_plot[var1],
        centroides_plot[var2],
        c='red',
        marker='X',
        s=250,
        label='Centróides'
      )

      for i, row in centroides_plot.iterrows():
        range_var1 = self.dados_cluster[var1].max() - self.dados_cluster[var1].min()
        range_var2 = self.dados_cluster[var2].max() - self.dados_cluster[var2].min()

        offset_x = range_var1 * 0.03 if range_var1 > 0 else 0.7
        offset_y = range_var2 * 0.003 if range_var2 > 0 else 0.15

        plt.text(
          row[var1] + offset_x,
          row[var2] + offset_y,
          nomes_clusters.get(i, f'Cluster {i}'),
          fontsize=10,
          fontweight='bold',
          color='darkred',
          bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8, edgecolor='red'),
          va='center'
        )

      handles, labels = plt.gca().get_legend_handles_labels()

      novos_labels = [
        nomes_clusters.get(int(l), l) if l.isdigit() else l
        for l in labels
      ]

      handles_ordenados = []
      labels_ordenados = []

      for nome in nomes_clusters.values():
        if nome in novos_labels:
          idx = novos_labels.index(nome)
          handles_ordenados.append(handles[idx])
          labels_ordenados.append(nome)

      if handles_ordenados:
        plt.legend(handles_ordenados, labels_ordenados, title="Clusters")

      plt.title(f'Clusters {posicao}: {mapa_variaveis.get(var1, var1)} vs {mapa_variaveis.get(var2, var2)}', fontsize=16)
      plt.xlabel(mapa_variaveis.get(var1, var1), fontsize=14)
      plt.ylabel(mapa_variaveis.get(var2, var2), fontsize=14)

      plt.grid(True, alpha=0.3)
      plt.tight_layout()

      plt.savefig(os.path.join(BASE_DIR, 'clusters_kmeans.png'), dpi=300, bbox_inches='tight')
      #plt.show()
      plt.close()

    except Exception as e:
      print(f"Erro ao gerar gráfico: {e}")

  return kmeans_clusters

ClusterBrasileirao.aplicar_kmeans = aplicar_kmeans

In [ ]:
def identificar_perfis_clusters_centroides(self):
  nomes_clusters = {}

  posicao = self.dados_cluster['Posicao_Real'].iloc[0]

  centroides_padronizados = self.scaler.transform(self.centroides)
  centroides_df = pd.DataFrame(centroides_padronizados, columns=self.variaveis_metricas, index=self.centroides.index)

  scores_por_cluster = {}

  for cluster_id in centroides_df.index:
    centroide = centroides_df.loc[cluster_id]

    if posicao == 'Goleiro':
      score_saída = (
        centroide['totalPass'] * 2 +
        centroide['accuratePass'] * 2
      )

      score_defesa = (
        centroide['ballRecovery'] * 3 +
        centroide['aerialWon'] * 2
      )

      score_equilibrado = (
        centroide['totalPass'] * 1 +
        centroide['ballRecovery'] * 1
      )

      scores_por_cluster[cluster_id] = {
        'Saída de Jogo': score_saída,
        'Defesa': score_defesa,
        'Equilibrado': score_equilibrado
      }

    elif posicao == 'Zagueiro':
      score_aereo = (
        centroide['aerialWon'] * 3 +
        centroide['totalTackle'] * 1
      )

      score_bola_chao = (
        centroide['interceptionWon'] * 2 +
        centroide['ballRecovery'] * 2 +
        centroide['totalPass'] * 1
      )

      score_equilibrado = (
        centroide['aerialWon'] * 1 +
        centroide['interceptionWon'] * 1
      )

      scores_por_cluster[cluster_id] = {
        'Aéreo': score_aereo,
        'Bola no Chão': score_bola_chao,
        'Equilibrado': score_equilibrado
      }

    elif posicao == 'Volante':
      score_defensivo = (
        centroide['totalTackle'] * 3 +
        centroide['interceptionWon'] * 2 +
        centroide['ballRecovery'] * 2
      )

      score_organizador = (
        centroide['totalPass'] * 2 +
        centroide['accuratePass'] * 2
      )

      score_equilibrado = (
        centroide['totalTackle'] * 1 +
        centroide['totalPass'] * 1
      )

      scores_por_cluster[cluster_id] = {
        'Defensivo': score_defensivo,
        'Organizador': score_organizador,
        'Equilibrado': score_equilibrado
      }

    else:
      score_criador = (
        centroide['keyPass'] * 3 +
        centroide['goalAssist'] * 3 #+
        #centroide['bigChanceCreated'] * 2
      )

      score_finalizador = (
        centroide['goals'] * 3 +
        centroide['totalShots'] * 2 #+
        #centroide['onTargetScoringAttempt'] * 2
      )

      score_equilibrado = (
        centroide['totalPass'] * 1 +
        centroide['accuratePass'] * 1 +
        centroide['touches'] * 0.5
      )

      scores_por_cluster[cluster_id] = {
        'Criador': score_criador,
        'Finalizador': score_finalizador,
        'Equilibrado': score_equilibrado
      }

  perfis_disponiveis = list(list(scores_por_cluster.values())[0].keys())
  n_clusters = len(scores_por_cluster)

  ranking_por_perfil = {}
  for perfil in perfis_disponiveis:
    ranking = sorted(scores_por_cluster.items(), key=lambda x: x[1][perfil], reverse=True)
    ranking_por_perfil[perfil] = ranking

  perfis_usados = set()

  for perfil in perfis_disponiveis:
    if not ranking_por_perfil[perfil]:
      continue

    for cluster_id, _ in ranking_por_perfil[perfil]:
      if cluster_id not in nomes_clusters:
        nomes_clusters[cluster_id] = perfil
        perfis_usados.add(perfil)
        break

  for cluster_id in scores_por_cluster:
    if cluster_id not in nomes_clusters:
      scores_cluster = scores_por_cluster[cluster_id]

      melhor_perfil = None
      melhor_score = float('-inf')

      for perfil in perfis_disponiveis:
        score = scores_cluster[perfil]
        if score > melhor_score:
          melhor_score = score
          melhor_perfil = perfil

      nomes_clusters[cluster_id] = melhor_perfil

  for cluster_id in sorted(scores_por_cluster.keys()):
    print(f"Cluster {cluster_id}: {nomes_clusters[cluster_id]}")
    for key, value in scores_por_cluster[cluster_id].items():
      print(f"  - {key}: {value:.2f}")

  return nomes_clusters

ClusterBrasileirao.identificar_perfis_clusters_centroides = identificar_perfis_clusters_centroides

### 5.1.7. Variância ANOVA

In [ ]:
def comparar_clusters_anova(self):
    print("\n=== ANÁLISE DE VARIÂNCIA (ANOVA) ===")

    resultados_anova = []

    for variavel in self.variaveis_metricas:
        print(f"\n--- ANOVA para {variavel} ---")

        resultado = pg.anova(
            dv=variavel,
            between='cluster_kmeans',
            data=self.dados_cluster,
            detailed=True
        )

        print(resultado.round(4))

        linha_cluster = resultado[resultado['Source'] == 'cluster_kmeans']

        if not linha_cluster.empty:
            linha_cluster = linha_cluster.iloc[0]

            f_stat = linha_cluster['F']
            p_value = linha_cluster['p_unc']
            np2 = linha_cluster['np2']

            resultados_anova.append({
                'variavel': variavel,
                'SS': linha_cluster['SS'],
                'DF': linha_cluster['DF'],
                'MS': linha_cluster['MS'],
                'F_stat': f_stat,
                'p_value': p_value,
                'np2': np2,
                'significativo': p_value < 0.05
            })

            print(f"F_stat: {f_stat:.3f}")
            print(f"p_value: {p_value:.4f}")
            print(f"np2: {np2:.3f}")
            print(f"Significativo: {'Sim' if p_value < 0.05 else 'Não'}")

    df_anova = pd.DataFrame(resultados_anova)
    self.df_anova = df_anova

    if not df_anova.empty:
        df_anova = df_anova.sort_values(by='np2', ascending=False)

        mais_discriminante = df_anova.iloc[0]['variavel']
        print(f"\nVariável mais discriminante: {mais_discriminante}")

    return df_anova

ClusterBrasileirao.comparar_clusters_anova = comparar_clusters_anova

In [ ]:
def plot_anova(self, df_anova):

    if df_anova.empty:
        print("DataFrame ANOVA está vazio.")
        return

    df_anova = df_anova.sort_values(by='F_stat', ascending=False)

    labels = df_anova['variavel'].map(mapa_variaveis).fillna(df_anova['variavel'])

    fig, ax1 = plt.subplots(figsize=(12, 6))

    ax1.bar(labels, df_anova['F_stat'], color='steelblue')
    ax1.set_ylabel('F_stat')
    ax1.set_xlabel('Variáveis')
    plt.xticks(rotation=45, ha='right')

    ax2 = ax1.twinx()

    ax2.plot(
        labels,
        df_anova['p_value'],
        color='orange',
        marker='o',
        linewidth=2,
        label='p-value'
    )

    ax2.set_yscale('log')

    ax2.yaxis.set_major_locator(ticker.LogLocator(base=10))
    ax2.yaxis.set_major_formatter(ticker.FuncFormatter(lambda y, _: f'{y:.0e}' if y < 0.01 else f'{y:.2f}'))

    ax2.axhline(0.05, color='red', linestyle='--', label='p = 0.05')

    ax2.set_ylabel('p-value', rotation=270, labelpad=15)
    ax2.yaxis.set_label_position("right")

    ax2.legend(loc='center left', bbox_to_anchor=(0.87, 0.5)
)

    plt.title('ANOVA: F_stat e p_value por variável')
    plt.tight_layout()
    #plt.show()
    plt.close()

ClusterBrasileirao.plot_anova = plot_anova

### 5.1.8. Clusters no gráfico 3D

In [ ]:
def gerar_clusters_3d(self):
  if len(self.variaveis_metricas) >= 3:
    print("\n=== VISUALIZAÇÃO 3D DOS CLUSTERS ===")

    var1, var2, var3 = self.variaveis_metricas[:3]

    fig = px.scatter_3d(
      self.dados_cluster,
      x=var1,
      y=var2,
      z=var3,
      color='cluster_kmeans',
      text=self.dados_cluster['Jogador'],
      title=f'Clusters 3D: {var1} vs {var2} vs {var3}'
    )

    fig.write_html(os.path.join(BASE_DIR, 'clusters_3d.html'))
    print("Visualização 3D salva como 'clusters_3d.html'")

ClusterBrasileirao.gerar_clusters_3d = gerar_clusters_3d

### 5.1.9. Interpretar perfil dos clusters (jogadores)

In [ ]:
def interpretar_perfis_clusters(self):
  print("\n=== ANÁLISE DE PERFIS DOS CLUSTERS ===")

  n_clusters = self.dados_cluster['cluster_kmeans'].nunique()

  for cluster_id in sorted(self.dados_cluster['cluster_kmeans'].unique()):
    print(f"\n--- CLUSTER {cluster_id} ---")

    dados_cluster = self.dados_cluster[self.dados_cluster['cluster_kmeans'] == cluster_id]

    print(f"Tamanho: {len(dados_cluster)} jogadores")
    print(f"Percentual: {(len(dados_cluster)/len(self.dados_cluster)*100):.1f}%")

    # Usar minutesPlayed para ordenar em vez de rating
    if 'minutesPlayed' in dados_cluster.columns:
      top_jogadores = dados_cluster.nlargest(5, 'minutesPlayed')[['Jogador', 'Time', 'Posicao_Real', 'minutesPlayed']]
      print("\nTop 5 jogadores (minutos jogados):")
      for _, row in top_jogadores.iterrows():
        print(f"  {row['Jogador']} ({row['Posicao_Real']}) - {row['Time']}: {row['minutesPlayed']:.0f} min")
    else:
      top_jogadores = dados_cluster.head(5)[['Jogador', 'Time', 'Posicao_Real']]
      print("\nTop 5 jogadores:")
      for _, row in top_jogadores.iterrows():
        print(f"  {row['Jogador']} ({row['Posicao_Real']}) - {row['Time']}")

    print("\nCaracterísticas médias:")
    for var in ['totalPass', 'totalShots', 'goals', 'totalTackle', 'keyPass']:
      if var in dados_cluster.columns:
        media = dados_cluster[var].mean()
        print(f"  {var}: {media:.2f}")

    dist_posicoes = dados_cluster['Posicao_Real'].value_counts()
    print(f"\nPosições principais:")
    for posicao, count in dist_posicoes.head(5).items():
      percent = (count / len(dados_cluster)) * 100
      print(f"  {posicao}: {count} ({percent:.1f}%)")

ClusterBrasileirao.interpretar_perfis_clusters = interpretar_perfis_clusters

# 6. Identificar Reforços

## 6.1. Classe GapAnalysis

### 6.1.1. Inicialização da classe GapAnalysis

In [ ]:
class GapAnalysis:
  def __init__(self, dataframe, clusters_result):
    self.dataframe = dataframe
    self.clusters_result = clusters_result
    self.dados_cluster = clusters_result.dados_cluster
    self.gaps_identificados = None
    self.benchmarks = None
    self.recomendacoes = None

### 6.1.2. Definição de benchmarks

In [ ]:
def definir_benchmarks(self, criterio='top_times', n_benchmark=5, times_especificos=None):
  print("=== DEFININDO BENCHMARKS ===")

  colunas_existentes = [col for col in regras_gap_agregacao.keys() if col in self.dataframe.columns]
  agregacoes_filtradas = {k: v for k, v in regras_gap_agregacao.items() if k in colunas_existentes}

  print(f"Colunas disponíveis para agregação: {colunas_existentes}")

  df_agregado = self.dataframe.groupby(['Time', 'Posicao_Real']).agg(agregacoes_filtradas).reset_index()

  if criterio == 'top_times':
    rating_por_time = df_agregado.groupby('Time')['rating'].mean().sort_values(ascending=False)
    top_times = rating_por_time.head(n_benchmark).index.tolist()
    print(f"Top {n_benchmark} times por rating: {top_times}")

  elif criterio == 'times_especificos' and times_especificos:
    top_times = times_especificos
    print(f"Times específicos definidos: {top_times}")

  else:
    rating_por_time = df_agregado.groupby('Time')['rating'].mean().sort_values(ascending=False)
    top_times = rating_por_time.head(5).index.tolist()
    print(f"Top 5 times por rating (padrão): {top_times}")

  self.benchmarks = df_agregado[df_agregado['Time'].isin(top_times)].copy()
  print(f"Benchmarks definidos: {len(self.benchmarks)} registros de {len(top_times)} times")

  return self.benchmarks

GapAnalysis.definir_benchmarks = definir_benchmarks

### 6.1.3. Cálcular desvios

In [ ]:
def calcular_desvios_por_metrica(self, time_alvo, posicao_alvo=None):
  print(f"\n=== CALCULANDO DESVIOS PARA TIME: {time_alvo} ===")

  if self.benchmarks is None:
    self.definir_benchmarks()

  df_time = self.dataframe[self.dataframe['Time'] == time_alvo].copy()

  if posicao_alvo:
    df_time = df_time[df_time['Posicao_Real'] == posicao_alvo]
    benchmarks_pos = self.benchmarks[self.benchmarks['Posicao_Real'] == posicao_alvo]
  else:
    benchmarks_pos = self.benchmarks

  if df_time.empty:
    print(f"Erro: Time '{time_alvo}' não encontrado no dataset.")
    return None

  colunas_existentes = [col for col in regras_gap_agregacao.keys() if col in df_time.columns]
  agregacoes_filtradas = {k: v for k, v in regras_gap_agregacao.items() if k in colunas_existentes}

  agregacao_time = df_time.groupby('Posicao_Real').agg(agregacoes_filtradas).reset_index()

  benchmark_medias = benchmarks_pos.groupby('Posicao_Real').agg(agregacoes_filtradas).reset_index()

  desvios = []
  metricas_analise = [col for col in ['rating', 'totalTackle', 'interceptionWon', 'ballRecovery',
                      'keyPass', 'goalAssist', 'totalPass', 'accuratePass',
                      'aerialWon', 'onTargetScoringAttempt', 'bigChanceCreated', 'expectedGoals']
                      if col in colunas_existentes]

  for posicao in agregacao_time['Posicao_Real'].unique():
    dados_pos = agregacao_time[agregacao_time['Posicao_Real'] == posicao].iloc[0]
    benchmark_pos = benchmark_medias[benchmark_medias['Posicao_Real'] == posicao]

    if benchmark_pos.empty:
      print(f"Aviso: Sem benchmark para posição {posicao}")
      continue

    benchmark_pos = benchmark_pos.iloc[0]

    for metrica in metricas_analise:
      if metrica in dados_pos.index and metrica in benchmark_pos.index:
        valor_time = dados_pos[metrica]
        valor_benchmark = benchmark_pos[metrica]

        if valor_benchmark != 0:
          gap_percentual = ((valor_time - valor_benchmark) / valor_benchmark) * 100
        else:
          gap_percentual = 0

        gap_absoluto = valor_time - valor_benchmark

        desvios.append({
            'Time': time_alvo,
            'Posicao': posicao,
            'Metrica': metrica,
            'Valor_Time': valor_time,
            'Valor_Benchmark': valor_benchmark,
            'Gap_Absoluto': gap_absoluto,
            'Gap_Percentual': gap_percentual,
            'Gap_Severidade': self.classificar_severidade(gap_percentual)
        })

  self.gaps_identificados = pd.DataFrame(desvios)

  if not self.gaps_identificados.empty:
    print(f"\nDesvios calculados: {len(self.gaps_identificados)} comparações")
    print("\nMaiores gaps (negativos = deficiência):")
    top_gaps = self.gaps_identificados.nsmallest(10, 'Gap_Percentual')
    for _, row in top_gaps.iterrows():
      print(f"{row['Posicao']} - {row['Metrica']}: {row['Gap_Percentual']:.1f}% "
            f"(Time: {row['Valor_Time']:.2f} vs Benchmark: {row['Valor_Benchmark']:.2f})")

  return self.gaps_identificados

GapAnalysis.calcular_desvios_por_metrica = calcular_desvios_por_metrica

### 6.1.4. Classificar severidade

In [ ]:
def classificar_severidade(self, gap_percentual):
  """Classifica a severidade do gap."""
  if gap_percentual <= -30:
    return 'Crítico'
  elif gap_percentual <= -15:
    return 'Alto'
  elif gap_percentual <= -5:
    return 'Moderado'
  elif gap_percentual < 5:
    return 'Adequado'
  else:
    return 'Acima da média'

GapAnalysis.classificar_severidade = classificar_severidade

### 6.1.5. Cálcular pontuação

In [ ]:
def calcular_pontuacao_gaps(self, time_alvo, posicao_alvo=None):

  print(f"\n=== CALCULANDO PONTUAÇÃO DE GAPS ===")

  if self.gaps_identificados is None:
    self.calcular_desvios_por_metrica(time_alvo, posicao_alvo)

  pontuacoes = []

  for posicao in self.gaps_identificados['Posicao'].unique():
    if posicao_alvo and posicao != posicao_alvo:
      continue

    gaps_pos = self.gaps_identificados[self.gaps_identificados['Posicao'] == posicao]
    pesos = pesos_por_posicao.get(posicao, {})

    pontuacao_total = 0
    peso_total = 0

    for _, row in gaps_pos.iterrows():
      metrica = row['Metrica']
      gap = row['Gap_Percentual']
      peso = pesos.get(metrica, 1)

      if gap < 0:
        pontuacao_metrica = abs(gap) * peso
      else:
        pontuacao_metrica = 0

      pontuacao_total += pontuacao_metrica
      peso_total += peso

    if peso_total > 0:
      pontuacao_normalizada = pontuacao_total / peso_total
    else:
      pontuacao_normalizada = 0

    pontuacoes.append({
        'Time': time_alvo,
        'Posicao': posicao,
        'Pontuacao_Gap': pontuacao_normalizada,
        'N_Metricas_Avaliadas': len(gaps_pos),
        'Gaps_Criticos': len(gaps_pos[gaps_pos['Gap_Severidade'] == 'Crítico']),
        'Gaps_Altos': len(gaps_pos[gaps_pos['Gap_Severidade'] == 'Alto'])
    })

  df_pontuacoes = pd.DataFrame(pontuacoes).sort_values('Pontuacao_Gap', ascending=False)

  print("\nPontuação de Gaps por Posição:")
  for _, row in df_pontuacoes.iterrows():
    print(f"{row['Posicao']}: {row['Pontuacao_Gap']:.1f} pontos "
          f"({row['Gaps_Criticos']} críticos, {row['Gaps_Altos']} altos)")

  return df_pontuacoes

GapAnalysis.calcular_pontuacao_gaps = calcular_pontuacao_gaps

### 6.1.6. Mapear GAPS para os clusters

In [ ]:
def mapear_gaps_para_clusters(self, time_alvo, posicao_alvo=None):
  print(f"\n=== MAPEANDO GAPS PARA PERFIS DE CLUSTERS ===")

  if self.gaps_identificados is None:
    self.calcular_desvios_por_metrica(time_alvo, posicao_alvo)

  pontuacoes = self.calcular_pontuacao_gaps(time_alvo, posicao_alvo)

  if self.clusters_result is None or self.clusters_result.dados_cluster is None:
    print("Erro: Resultados do clustering não disponíveis.")
    return None

  if hasattr(self.clusters_result, 'centroides') and self.clusters_result.centroides is not None:
    nomes_clusters = self.clusters_result.identificar_perfis_clusters_centroides()
  else:
    print("Aviso: Centróides não disponíveis, usando IDs numéricos")
    nomes_clusters = {i: f'Cluster {i}' for i in range(3)}

  recomendacoes = []

  for _, row in pontuacoes.iterrows():
    posicao = row['Posicao']
    pontuacao_gap = row['Pontuacao_Gap']

    gaps_pos = self.gaps_identificados[self.gaps_identificados['Posicao'] == posicao]

    gaps_ordenados = gaps_pos.sort_values('Gap_Percentual')
    top_gaps = gaps_ordenados.head(3)

    metricas_deficientes = top_gaps['Metrica'].tolist()

    perfil_recomendado = self.determinar_perfil_recomendado(posicao, metricas_deficientes, nomes_clusters)

    recomendacoes.append({
        'Time': time_alvo,
        'Posicao': posicao,
        'Pontuacao_Gap': pontuacao_gap,
        'Prioridade': self._classificar_prioridade(pontuacao_gap),
        'Metricas_Deficientes': metricas_deficientes,
        'Perfil_Recomendado': perfil_recomendado,
        'Justificativa': self._gerar_justificativa(posicao, metricas_deficientes, gaps_pos)
    })

  self.recomendacoes = pd.DataFrame(recomendacoes).sort_values('Pontuacao_Gap', ascending=False)

  print("\n=== RECOMENDAÇÕES DE CONTRATAÇÃO ===")
  for _, row in self.recomendacoes.iterrows():
    print(f"\n{row['Posicao']} - Prioridade: {row['Prioridade']} ({row['Pontuacao_Gap']:.1f} pts)")
    print(f"Perfil Recomendado: {row['Perfil_Recomendado']}")
    print(f"Métricas Deficientes: {', '.join(row['Metricas_Deficientes'])}")
    print(f"Justificativa: {row['Justificativa']}")

  return self.recomendacoes

GapAnalysis.mapear_gaps_para_clusters = mapear_gaps_para_clusters

### 6.1.7. Perfil recomendado

In [ ]:
def determinar_perfil_recomendado(self, posicao, metricas_deficientes, nomes_clusters):

  contagem_perfis = {}
  mapeamento_pos = mapeamento_perfis.get(posicao, {})

  for metrica in metricas_deficientes:
    perfil = mapeamento_pos.get(metrica, 'Equilibrado')
    contagem_perfis[perfil] = contagem_perfis.get(perfil, 0) + 1

  if contagem_perfis:
    perfil_recomendado = max(contagem_perfis, key=contagem_perfis.get)
  else:
    perfil_recomendado = 'Equilibrado'

  if perfil_recomendado in nomes_clusters.values():
    return perfil_recomendado
  else:
    return list(nomes_clusters.values())[0] if nomes_clusters else 'Cluster 0'

GapAnalysis.determinar_perfil_recomendado = determinar_perfil_recomendado

### 6.1.8. Gerar Prioridade e Justificativa

In [ ]:
def _classificar_prioridade(self, pontuacao_gap):

  if pontuacao_gap >= 30:
    return 'Alta'
  elif pontuacao_gap >= 15:
    return 'Média'
  else:
    return 'Baixa'

GapAnalysis._classificar_prioridade = _classificar_prioridade

In [ ]:
def _gerar_justificativa(self, posicao, metricas_deficientes, gaps_pos):

  metricas_nomes = [mapa_variaveis.get(m, m) for m in metricas_deficientes]

  gaps_top = gaps_pos[gaps_pos['Metrica'].isin(metricas_deficientes)].sort_values('Gap_Percentual')

  justificativa = f"O time apresenta deficiências em {', '.join(metricas_nomes[:2])} "

  if not gaps_top.empty:
    gap_principal = gaps_top.iloc[0]
    justificativa += f"({gap_principal['Gap_Percentual']:.1f}% abaixo do benchmark). "

  justificativa += "Este perfil de jogador ajudaria a corrigir estas lacunas táticas."

  return justificativa

GapAnalysis._gerar_justificativa = _gerar_justificativa

### 6.1.9. Gerar Relatório (Gráfico)

In [ ]:
def gerar_relatorio_visual(self, time_alvo, output_dir=None):

  if self.gaps_identificados is None or self.gaps_identificados.empty:
    print("Erro: Gaps não identificados. Execute calcular_desvios_por_metrica primeiro.")
    return

  if output_dir is None:
    output_dir = BASE_DIR

  print(f"\n=== GERANDO RELATÓRIO VISUAL ===")

  fig, axes = plt.subplots(2, 2, figsize=(16, 12))

  pivot_gaps = self.gaps_identificados.pivot(index='Posicao', columns='Metrica', values='Gap_Percentual')

  if not pivot_gaps.empty:
    sns.heatmap(pivot_gaps, annot=True, fmt='.1f', cmap='RdYlGn', center=0,
                ax=axes[0, 0], cbar_kws={'label': 'Gap %'})
    axes[0, 0].set_title('Heatmap de Gaps por Posição e Métrica', fontsize=14, fontweight='bold')
    axes[0, 0].set_xlabel('Métricas', fontsize=12)
    axes[0, 0].set_ylabel('Posições', fontsize=12)

  top_gaps = self.gaps_identificados.nsmallest(10, 'Gap_Percentual')
  if not top_gaps.empty:
    labels = [f"{row['Posicao']}\n{mapa_variaveis.get(row['Metrica'], row['Metrica'])}"
              for _, row in top_gaps.iterrows()]
    axes[0, 1].barh(range(len(top_gaps)), top_gaps['Gap_Percentual'], color='coral')
    axes[0, 1].set_yticks(range(len(top_gaps)))
    axes[0, 1].set_yticklabels(labels, fontsize=9)
    axes[0, 1].set_xlabel('Gap Percentual (%)', fontsize=12)
    axes[0, 1].set_title('Top 10 Maiores Gaps (Deficiências)', fontsize=14, fontweight='bold')
    axes[0, 1].axvline(x=0, color='black', linestyle='--', alpha=0.5)

  if not self.gaps_identificados.empty:
    severidade_counts = self.gaps_identificados['Gap_Severidade'].value_counts()
    axes[1, 0].pie(severidade_counts.values, labels=severidade_counts.index,
                  autopct='%1.1f%%', colors=['red', 'orange', 'yellow', 'green', 'blue'])
    axes[1, 0].set_title('Distribuição de Severidade de Gaps', fontsize=14, fontweight='bold')

  if self.recomendacoes is not None and not self.recomendacoes.empty:
    axes[1, 1].barh(self.recomendacoes['Posicao'], self.recomendacoes['Pontuacao_Gap'],
                    color='steelblue')
    axes[1, 1].set_xlabel('Pontuação de Gap', fontsize=12)
    axes[1, 1].set_title('Pontuação de Gaps por Posição', fontsize=14, fontweight='bold')
    axes[1, 1].grid(axis='x', alpha=0.3)

  plt.tight_layout()
  output_path = os.path.join(output_dir, f'gap_analysis_{time_alvo.replace(" ", "_")}.png')
  plt.savefig(output_path, dpi=300, bbox_inches='tight')
  print(f"Relatório visual salvo em: {output_path}")
  plt.close()

  if self.recomendacoes is not None and not self.recomendacoes.empty:
    fig, ax = plt.subplots(figsize=(12, 8))

    rec_ordenadas = self.recomendacoes.sort_values('Pontuacao_Gap', ascending=True)

    colors = ['red' if p == 'Alta' else 'orange' if p == 'Média' else 'green'
              for p in rec_ordenadas['Prioridade']]

    bars = ax.barh(rec_ordenadas['Posicao'], rec_ordenadas['Pontuacao_Gap'], color=colors, alpha=0.7)

    for i, (bar, perfil) in enumerate(zip(bars, rec_ordenadas['Perfil_Recomendado'])):
      width = bar.get_width()
      ax.text(width + 1, bar.get_y() + bar.get_height()/2,
              f' {perfil}', ha='left', va='center', fontsize=10, fontweight='bold')

    ax.set_xlabel('Pontuação de Gap', fontsize=14)
    ax.set_title(f'Recomendações de Contratação - {time_alvo}', fontsize=16, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)

    legend_elements = [plt.Rectangle((0,0),1,1, color='red', alpha=0.7, label='Prioridade Alta'),
                      plt.Rectangle((0,0),1,1, color='orange', alpha=0.7, label='Prioridade Média'),
                      plt.Rectangle((0,0),1,1, color='green', alpha=0.7, label='Prioridade Baixa')]
    ax.legend(handles=legend_elements, loc='upper right')

    plt.tight_layout()
    output_path_rec = os.path.join(output_dir, f'recomendacoes_{time_alvo.replace(" ", "_")}.png')
    plt.savefig(output_path_rec, dpi=300, bbox_inches='tight')
    print(f"Gráfico de recomendações salvo em: {output_path_rec}")
    plt.close()

  return output_path

GapAnalysis.gerar_relatorio_visual = gerar_relatorio_visual

### 6.1.10. Identifcar Reforços do Elenco

In [ ]:
def identificar_reforcos_elenco(self, time_alvo=None, posicao_alvo=None):
  print("\n" + "="*60)
  print("GAP ANALYSIS - IDENTIFICAÇÃO DE LACUNAS NO ELBENCO")
  print("="*60)

  if time_alvo is None:
    time_counts = self.dataframe['Time'].value_counts()
    time_alvo = time_counts.index[0]
    print(f"Time alvo não especificado. Usando time com mais jogadores: {time_alvo}")

  gap_analyzer = GapAnalysis(self.dataframe, self)

  gap_analyzer.definir_benchmarks(criterio='top_times', n_benchmark=5)

  gap_analyzer.calcular_desvios_por_metrica(time_alvo, posicao_alvo)

  gap_analyzer.calcular_pontuacao_gaps(time_alvo, posicao_alvo)

  recomendacoes = gap_analyzer.mapear_gaps_para_clusters(time_alvo, posicao_alvo)

  gap_analyzer.gerar_relatorio_visual(time_alvo)

  return gap_analyzer

GapAnalysis.identificar_reforcos_elenco = identificar_reforcos_elenco

# 7. Main

In [ ]:
def main(posicao_alvo=None, min_jogos=CONFIG['MIN_JOGOS']):
  if posicao_alvo:
    print(f"\n{'='*60}")
    print(f"Clustering posição: {posicao_alvo}")
    print(f"{'='*60}\n")
  else:
    print(f"\n{'='*60}")
    print(f"Clustering geral")
    print(f"{'='*60}\n")

  cluster_resultado = ClusterBrasileirao(CONFIG['DATASET_PATH'])

  try:
    dados_metricos_result = cluster_resultado.gerar_base(min_jogos=min_jogos, posicao_alvo=posicao_alvo)
  except Exception as e:
    print('Erro ao gerar dados métricos: ' + tb.format_exc())
    return None

  if len(cluster_resultado.dados_cluster) < 10:
    print(f"Jogadores insuficientes para clustering: {len(cluster_resultado.dados_cluster)}")
    return None

  cluster_resultado.explorar_variaveis_dataframe()

  dados_padronizados = cluster_resultado.padronizar_dados(metodo=CONFIG['METODO_PADRONIZAR'])

  if posicao_alvo:
    if posicao_alvo == 'Goleiro':
        k_ideal = 3
        print(f"\nUsando k=3 para posição {posicao_alvo}")
    elif posicao_alvo == 'Zagueiro':
        k_ideal = 3
        print(f"\nUsando k=3 para posição {posicao_alvo}")
    elif posicao_alvo == 'Volante':
        k_ideal = 3
        print(f"\nUsando k=3 para posição {posicao_alvo}")
    else:
        k_ideal = cluster_resultado.analisar_clusters(max_clusters=CONFIG['N_CLUSTERS_MAX'], random_state=CONFIG['RANDOM_STATE'])

        if k_ideal > 3:
            k_ideal = 3
            print(f"\nLimitando k para 3 (clustering por posição)")
  else:
    k_ideal = cluster_resultado.analisar_clusters(max_clusters=CONFIG['N_CLUSTERS_MAX'], random_state=CONFIG['RANDOM_STATE'])

  print(f"\n{'='*60}")
  print(f"Número de clusters: {k_ideal}")
  print(f"{'='*60}\n")

  if not posicao_alvo:
    print("\n" + "="*60)
    print("TESTANDO CLUSTERING HIERÁRQUICO")
    print("="*60)

  cluster_resultado.aplicar_hierarquico(n_clusters=k_ideal, metodo_linkage='complete')
  cluster_resultado.aplicar_hierarquico(n_clusters=k_ideal, metodo_linkage='average')

  print("\n" + "="*60)
  print("TESTANDO K-MEANS")
  print("="*60)
  cluster_resultado.aplicar_kmeans(n_clusters=k_ideal)

  print("\n" + "="*60)
  print("ANOVA PARA K-MEANS")
  print("="*60)
  cluster_resultado.comparar_clusters_anova()
  cluster_resultado.plot_anova(cluster_resultado.df_anova)

  gap_resultado = GapAnalysis(cluster_resultado.dados_cluster, cluster_resultado)

  if not posicao_alvo:
    print("\n" + "="*60)
    print("GAP ANALYSIS - IDENTIFICAÇÃO DE LACUNAS")
    print("="*60)
    gap_resultado.identificar_reforcos_elenco()
  else:
    print("\n" + "="*60)
    print("GAP ANALYSIS - IDENTIFICAÇÃO DE LACUNAS")
    print("="*60)
    gap_resultado.identificar_reforcos_elenco('Botafogo', posicao_alvo=posicao_alvo)

  return cluster_resultado

if __name__ == "__main__":
  main('Meia Central')


Clustering posição: Meia Central

Usando métricas específicas para Meia Central: ['keyPass', 'goalAssist', 'goals', 'totalShots', 'totalPass', 'accuratePass', 'touches']
Total de jogadores clusterizados (Meia Central): 260
********* Análise Exploratória *********
               mean    std    min    max
keyPass        1.04   0.57   0.08   3.00
goalAssist     1.68   1.95   0.00   7.00
goals          2.93   3.45   0.00  13.00
totalShots     1.39   0.55   0.38   3.00
totalPass     25.13  12.60   7.31  78.29
accuratePass  20.79  11.83   4.50  71.71
touches       37.42  13.93  13.50  85.86




Padronização StandardScaler aplicada
Analisando número ideal de clusters

Nº sugerido de clusters (cotovelo): 4
Nº sugerido de clusters (Silhouette): 3
Elbow: 4
Silhouette: 3

Limitando k para 3 (clustering por posição)

Número de clusters: 3


********** Algoritmo Hierárquico (complete linkage) **********
Coeficientes de aglomeração (complete): [np.float64(1.7080053757212363), np.float64(1.305042738